# PIIS: Telecom Incident Triage and Evidence Investigation

This notebook demonstrates the end-to-end analytical workflow of the **Production Incident Intelligence System (PIIS)**.

### Workflow Objectives
1. Ingest multi-vendor telecom incident envelopes (ServiceNow, Jira).
2. Execute multi-domain triage classification and severity prediction.
3. Perform causal telemetry anomaly detection and topological blast radius analysis.
4. Run hybrid evidence retrieval (Okapi BM25 + dense semantic embeddings).
5. Review synthesized hypotheses, evidence registers, and runbook recommendations.


In [ ]:
import sys
import os
from pprint import pprint
from datetime import datetime, timezone

# Add src directory to Python path
sys.path.insert(0, os.path.abspath('../src'))

from piis.pipeline.unified import UnifiedIngestionPipeline
from piis.integrations.contracts import IngestionEnvelope
from piis.integrations.itsm.servicenow import ServiceNowAdapter
from piis.integrations.telemetry.prometheus import PrometheusAdapter
from piis.integrations.alarms.oss_nms import OSSAlarmAdapter

print('PIIS core modules successfully imported.')


## 1. Pipeline Initialization and Adapter Registration
PIIS uses a unified ingestion pipeline that standardizes multi-source telecom payloads into canonical envelopes.


In [ ]:
pipeline = UnifiedIngestionPipeline()
pipeline.register_adapter(ServiceNowAdapter())
pipeline.register_adapter(PrometheusAdapter())
pipeline.register_adapter(OSSAlarmAdapter(adapter_id='ALM-NOK', source_system='NokiaOSS', vendor_hint='NOKIA'))

print('Registered Adapters:', [a['adapter_id'] for a in pipeline.list_adapters()])


## 2. Ingesting a Canonical Telecom Incident
Simulate an incoming ServiceNow ticket reporting an RF feeder mismatch on a 5G gNodeB.


In [ ]:
now_iso = datetime.now(timezone.utc).isoformat()
ticket = {
    'sys_id': 'a98df7b01b827300c0147596cc4bcb11',
    'number': 'INC0089201',
    'short_description': 'Sector degraded: High VSWR on 5G NR Cell-1',
    'description': 'NOC telemetry indicates RF feeder mismatch and VSWR > 1.8 on gNodeB GNB-RAN-701.',
    'priority': '2',
    'category': 'RAN',
    'u_network_domain': 'RAN',
    'cmdb_ci': 'GNB-RAN-701',
    'state': '1',
    'sys_created_on': now_iso,
    'u_affected_subscribers': 1420,
    'u_fault_category': 'antenna_vswr_fault'
}

envelope = IngestionEnvelope.create(
    source_system='ServiceNow',
    source_type='OPERATOR_INTERNAL',
    raw_payload=ticket,
    external_event_id=ticket['number'],
    observation_timestamp=now_iso
)
result = pipeline.ingest_envelope(envelope)
print('Ingestion Status:', result.status)
print('Canonical Incident ID:', result.canonical_id)


## 3. Automated Evidence Grounding and Investigation Execution
Trigger the multi-stage investigation engine to analyze the incident against historical runbooks and active alarms.


In [ ]:
ctx = pipeline.investigate_incident(result.canonical_id)
print('Incident Title:', ctx.title)
print('Domain:', ctx.network_domain)
print('Hypotheses Count:', len(ctx.hypotheses))
print('Recommendations Count:', len(ctx.recommendations))
print('Epistemic Status:', ctx.epistemic_status)


## 4. Ranked Hypotheses and Diagnostic Recommendations
Inspect the synthesized hypotheses and human-reviewable diagnostic steps.


In [ ]:
for idx, hyp in enumerate(ctx.hypotheses, 1):
    print(f'\nHypothesis #{idx} (Status: {hyp.status}, Confidence Proxy: {hyp.confidence_proxy}):')
    print(f'  Description: {hyp.description}')
    print(f'  Rationale: {hyp.confidence_rationale}')

print('\n--- Recommended Diagnostic Actions ---')
for rec in ctx.recommendations:
    print(f'- Action: {rec.description}')
    print(f'  Reason: {rec.reason}')
    print(f'  Runbook: {rec.source_runbook_id}')
    print(f'  Approval Required: {rec.human_approval_required}')
